# JupyterLab setup

This workspace and your settings survive container replacement in the Docker volume. Start by running the assessment below. It reads hardware details without allocating a workload.

Edit **lab_settings.py** in the file browser, save, then rerun setup to change presets and budgets.

In [ ]:
%run setup_lab.py

## Read the plan

CPU counts include Docker quota and affinity. Memory reports distinguish `/proc` total, effective limit, and current available capacity. NVIDIA VRAM is reported per physical device. Unavailable NVIDIA reporting does not mean the host lacks a GPU.

In [ ]:
from IPython.display import display
import pandas as pd

display(pd.DataFrame([report['hardware']['cpu']]))
display(pd.DataFrame([{k: round(v / 1024**3, 2) if v is not None else None
                       for k, v in report['hardware']['memory'].items()}]).add_suffix(' (GiB)'))
display(pd.DataFrame(report['hardware']['gpu']['devices']))
display(pd.DataFrame([{k: v for k, v in report['plan'].items() if k not in ('options', 'gpu_budgets')}]))

## Optional local CPU worker pool

Set `START_CLUSTER = True` to start the planned Dask processes. This is opt-in because large servers may start many processes. Dask memory limits apply to workers, not arbitrary notebook allocations. GPU budgets are advisory; this pool does not schedule GPU workloads. For ordinary NumPy/Pandas work, no cluster is required.

In [ ]:
START_CLUSTER = False
if START_CLUSTER:
    from labkit import start_cluster
    if 'client' in globals():
        client.close()
        cluster.close()
    cluster, client = start_cluster(report)
    display(client)
    print(client.gather(client.map(lambda x: x * x, range(10))))

## Close workers when finished

Close the pool before changing settings and creating a new pool. Applying settings changes the current kernel; other already-running kernels keep their own configuration.

In [ ]:
if 'client' in globals():
    client.close()
    cluster.close()

## Python packages and GPU computation

NumPy, Pandas, SciPy, Matplotlib, PyArrow, Dask, and widgets are preinstalled. NVIDIA monitoring works when the host provides the driver and NVIDIA Container Toolkit and the container is started with GPU access. CUDA frameworks such as PyTorch, TensorFlow, and CuPy are intentionally not bundled. Install the framework/version suited to your workload and driver using its official instructions. Restart the kernel after installation. For persistent extra packages use `%pip install --user PACKAGE`; record packages in your own requirements file.

A visible GPU is not a framework smoke test. MIG partitions and CUDA device restrictions can differ from this physical GPU report. A generic batch size cannot be derived from VRAM alone.